# TP 4 — Capture, assainissement et plan de replay inerte

**Public cible :** étudiant, technicien, analyste cyber ou NetOps débutant.  
**Durée :** 60 minutes. **Pré-requis :** savoir lire une adresse IP, un port UDP et une chronologie.

> Règle de départ : **capturer n'autorise pas à rejouer**. Une capture peut contenir des secrets, des identifiants et des données personnelles ; un replay peut déclencher des effets réels. Il faut une autorisation distincte pour chaque opération, un périmètre explicite et une destination de test isolée.

Ce notebook ne capture et ne transmet rien. Il fabrique en mémoire un **PCAP synthétique** Ethernet/IPv4/UDP, le parse, le filtre, l'anonymise, vérifie sa fidélité puis produit un **plan inerte** dont `execution_allowed=false`.


## Contrat d'apprentissage et déroulé

À la fin, vous saurez distinguer quatre objets : les octets capturés, la vue parsée, le PCAP assaini et le plan de replay.

1. **Prédire — 5 min :** quelles propriétés faut-il préserver sans conserver les identités ni le contenu ?
2. **Construire et parser — 15 min :** vérifier Ethernet, IPv4, UDP et leurs checksums.
3. **Filtrer et anonymiser — 15 min :** appliquer une règle auditable et reconstruire les paquets.
4. **Mesurer la fidélité — 10 min :** comparer tailles, horodatage, ordre et distribution des flows.
5. **Préparer sans exécuter — 10 min :** générer un plan de replay inerte et inspecter ses garde-fous.
6. **Expliquer — 5 min :** défendre ce que le TP prouve et ce qu'il ne prouve pas.

Avant d'exécuter, écrivez votre hypothèse : *anonymiser* signifie-t-il seulement remplacer les IP ? Que deviendraient les checksums après cette modification ?


In [ ]:
from __future__ import annotations

import base64
import hashlib
import html
import ipaddress
import json
import random
import struct
from collections import Counter

SEED = 20260730
rng = random.Random(SEED)
PCAP_MAGIC = 0xA1B2C3D4
PCAP_GLOBAL = struct.pack("<IHHIIII", PCAP_MAGIC, 2, 4, 0, 0, 65535, 1)

def internet_checksum(data: bytes) -> int:
    if len(data) % 2:
        data += b"\x00"
    total = sum(struct.unpack(f"!{len(data) // 2}H", data))
    while total >> 16:
        total = (total & 0xFFFF) + (total >> 16)
    return (~total) & 0xFFFF

def mac_bytes(address: str) -> bytes:
    return bytes.fromhex(address.replace(":", ""))

def build_udp_frame(src_mac: str, dst_mac: str, src_ip: str, dst_ip: str,
                    src_port: int, dst_port: int, payload: bytes, identification: int) -> bytes:
    src = ipaddress.IPv4Address(src_ip).packed
    dst = ipaddress.IPv4Address(dst_ip).packed
    udp_length = 8 + len(payload)
    udp_zero = struct.pack("!HHHH", src_port, dst_port, udp_length, 0)
    pseudo = src + dst + struct.pack("!BBH", 0, 17, udp_length)
    udp_sum = internet_checksum(pseudo + udp_zero + payload) or 0xFFFF
    udp = struct.pack("!HHHH", src_port, dst_port, udp_length, udp_sum)
    total_length = 20 + udp_length
    ip_zero = struct.pack("!BBHHHBBH4s4s", 0x45, 0, total_length, identification,
                          0x4000, 64, 17, 0, src, dst)
    ip_sum = internet_checksum(ip_zero)
    ip_header = struct.pack("!BBHHHBBH4s4s", 0x45, 0, total_length, identification,
                            0x4000, 64, 17, ip_sum, src, dst)
    ethernet = mac_bytes(dst_mac) + mac_bytes(src_mac) + struct.pack("!H", 0x0800)
    return ethernet + ip_header + udp + payload

def pack_pcap(records: list[tuple[int, bytes]]) -> bytes:
    body = bytearray()
    for timestamp_us, frame in records:
        seconds, micros = divmod(timestamp_us, 1_000_000)
        body.extend(struct.pack("<IIII", seconds, micros, len(frame), len(frame)))
        body.extend(frame)
    return PCAP_GLOBAL + bytes(body)

assert internet_checksum(b"123456789") == 0xF62A


## 1. Construire un vrai fichier PCAP, sans réseau

La capture contient trois flows fictifs : télémétrie, commandes et bruit de laboratoire. Les adresses, charges utiles et horodatages sont synthétiques. Le format PCAP, les en-têtes Ethernet/IPv4/UDP et les checksums sont en revanche encodés réellement avec `struct`.

**Question :** si le filtre retire le bruit, la capture assainie doit-elle conserver ses délais, ou seulement ceux des paquets retenus ? Nous choisirons la seconde définition et la rendrons explicite dans les preuves.


In [ ]:
BASE_US = 1_700_000_000_000_000
FLOW_SPECS = [
    {"name": "telemetry", "count": 14, "start_us": 0, "step_us": 75_000,
     "src_mac": "02:00:00:00:01:10", "dst_mac": "02:00:00:00:01:20",
     "src_ip": "10.42.0.10", "dst_ip": "10.42.0.20",
     "src_port": 41000, "dst_port": 5000, "payload_size": 72},
    {"name": "command", "count": 8, "start_us": 30_000, "step_us": 140_000,
     "src_mac": "02:00:00:00:01:20", "dst_mac": "02:00:00:00:01:10",
     "src_ip": "10.42.0.20", "dst_ip": "10.42.0.10",
     "src_port": 42000, "dst_port": 6000, "payload_size": 44},
    {"name": "lab-noise", "count": 6, "start_us": 10_000, "step_us": 190_000,
     "src_mac": "02:00:00:00:01:30", "dst_mac": "02:00:00:00:01:20",
     "src_ip": "10.42.0.30", "dst_ip": "10.42.0.20",
     "src_port": 43000, "dst_port": 9999, "payload_size": 31},
]

records = []
packet_id = 0
for flow in FLOW_SPECS:
    for sequence in range(flow["count"]):
        prefix = f"SYNTH|{flow['name']}|{sequence:02d}|".encode("ascii")
        padding = bytes(rng.randrange(0, 256) for _ in range(flow["payload_size"] - len(prefix)))
        payload = prefix + padding
        frame = build_udp_frame(
            flow["src_mac"], flow["dst_mac"], flow["src_ip"], flow["dst_ip"],
            flow["src_port"], flow["dst_port"], payload, 1000 + packet_id,
        )
        records.append((BASE_US + flow["start_us"] + sequence * flow["step_us"], frame))
        packet_id += 1
records.sort(key=lambda item: item[0])
pcap_bytes = pack_pcap(records)
assert pcap_bytes[:4] == b"\xd4\xc3\xb2\xa1"
assert len(records) == 28 and len(pcap_bytes) > 2_500
{"pcap_sha256": hashlib.sha256(pcap_bytes).hexdigest()[:16], "packets": len(records)}


## 2. Parser, vérifier, puis filtrer

Un filtre est une décision de périmètre, pas une décoration. Ici, il retient uniquement les ports de destination 5000 et 6000. Cette règle est versionnable, relisible et testable ; le flow de bruit sur 9999 doit disparaître.

Le parseur refuse les troncatures, longueurs incohérentes, protocoles inattendus et checksums invalides. En production, un outil éprouvé serait préférable ; l'objectif pédagogique est de voir les invariants sous l'interface graphique.


In [ ]:
def format_mac(raw: bytes) -> str:
    return ":".join(f"{octet:02x}" for octet in raw)

def parse_frame(frame: bytes) -> dict:
    assert len(frame) >= 42, "truncated Ethernet/IPv4/UDP frame"
    dst_mac, src_mac, ethertype = frame[:6], frame[6:12], struct.unpack("!H", frame[12:14])[0]
    assert ethertype == 0x0800, "only IPv4 is expected"
    version_ihl = frame[14]
    assert version_ihl == 0x45, "IPv4 options are outside this lab"
    ip_header = frame[14:34]
    total_length, identification, protocol = struct.unpack("!x x H H x x x B", ip_header[:10])
    assert protocol == 17 and total_length == len(frame) - 14
    assert internet_checksum(ip_header) == 0, "invalid IPv4 checksum"
    src_raw, dst_raw = ip_header[12:16], ip_header[16:20]
    udp_header = frame[34:42]
    src_port, dst_port, udp_length, udp_sum = struct.unpack("!HHHH", udp_header)
    payload = frame[42:14 + total_length]
    assert udp_length == 8 + len(payload) and udp_sum != 0
    pseudo = src_raw + dst_raw + struct.pack("!BBH", 0, 17, udp_length)
    assert internet_checksum(pseudo + udp_header + payload) == 0, "invalid UDP checksum"
    return {
        "src_mac": format_mac(src_mac), "dst_mac": format_mac(dst_mac),
        "src_ip": str(ipaddress.IPv4Address(src_raw)),
        "dst_ip": str(ipaddress.IPv4Address(dst_raw)),
        "src_port": src_port, "dst_port": dst_port,
        "identification": identification, "payload": payload,
        "frame": frame, "frame_length": len(frame),
    }

def parse_pcap(data: bytes) -> list[dict]:
    assert len(data) >= 24 and data[:24] == PCAP_GLOBAL, "unsupported PCAP header"
    offset = 24
    parsed = []
    while offset < len(data):
        assert offset + 16 <= len(data), "truncated PCAP record"
        seconds, micros, captured, original = struct.unpack_from("<IIII", data, offset)
        offset += 16
        assert micros < 1_000_000 and captured == original and offset + captured <= len(data)
        packet = parse_frame(data[offset:offset + captured])
        packet["timestamp_us"] = seconds * 1_000_000 + micros
        parsed.append(packet)
        offset += captured
    assert offset == len(data)
    return parsed

def flow_key(packet: dict) -> tuple:
    return (packet["src_ip"], packet["dst_ip"], packet["src_port"], packet["dst_port"], 17)

parsed_packets = parse_pcap(pcap_bytes)
FILTER_PORTS = frozenset({5000, 6000})
filtered_packets = [packet for packet in parsed_packets if packet["dst_port"] in FILTER_PORTS]
assert len(parsed_packets) == 28 and len(filtered_packets) == 22
assert {packet["dst_port"] for packet in filtered_packets} == FILTER_PORTS
assert len({flow_key(packet) for packet in filtered_packets}) == 2
Counter(packet["dst_port"] for packet in filtered_packets)


## 3. Anonymiser sans casser la structure

Le contrat d'assainissement remplace les IP, les MAC et chaque octet de charge utile. Les ports sont conservés pour l'analyse pédagogique. L'ordre, l'horodatage, la taille de chaque trame et le nombre de paquets par flow restent identiques. Les checksums sont recalculés après transformation.

Cette pseudonymisation n'est pas une garantie universelle d'anonymat : tailles, timing, ports et comportements peuvent encore ré-identifier un système. Pour une vraie diffusion, il faut une analyse de risque et une politique de minimisation adaptées.


In [ ]:
IP_MAP = {
    "10.42.0.10": "198.51.100.10",
    "10.42.0.20": "198.51.100.20",
    "10.42.0.30": "198.51.100.30",
}
MAC_MAP = {
    "02:00:00:00:01:10": "02:00:00:00:aa:10",
    "02:00:00:00:01:20": "02:00:00:00:aa:20",
    "02:00:00:00:01:30": "02:00:00:00:aa:30",
}

def scrub_payload(packet: dict, ordinal: int) -> bytes:
    seed_material = f"{SEED}|{flow_key(packet)}|{ordinal}".encode("utf-8")
    digest = hashlib.sha256(seed_material).digest()
    marker = b"SANITIZED|"
    stream = marker + digest * ((len(packet["payload"]) // len(digest)) + 1)
    return stream[:len(packet["payload"])]

sanitized_records = []
for ordinal, packet in enumerate(filtered_packets):
    clean_payload = scrub_payload(packet, ordinal)
    clean_frame = build_udp_frame(
        MAC_MAP[packet["src_mac"]], MAC_MAP[packet["dst_mac"]],
        IP_MAP[packet["src_ip"]], IP_MAP[packet["dst_ip"]],
        packet["src_port"], packet["dst_port"], clean_payload,
        packet["identification"],
    )
    sanitized_records.append((packet["timestamp_us"], clean_frame))
sanitized_pcap_bytes = pack_pcap(sanitized_records)
sanitized_packets = parse_pcap(sanitized_pcap_bytes)

size_fidelity = [p["frame_length"] for p in filtered_packets] == [p["frame_length"] for p in sanitized_packets]
time_fidelity = [p["timestamp_us"] for p in filtered_packets] == [p["timestamp_us"] for p in sanitized_packets]
order_fidelity = [p["dst_port"] for p in filtered_packets] == [p["dst_port"] for p in sanitized_packets]
original_flow_counts = sorted(Counter(flow_key(p) for p in filtered_packets).values())
sanitized_flow_counts = sorted(Counter(flow_key(p) for p in sanitized_packets).values())
flow_fidelity = original_flow_counts == sanitized_flow_counts
endpoints_changed = all(
    original["src_ip"] != clean["src_ip"] and original["dst_ip"] != clean["dst_ip"]
    and original["src_mac"] != clean["src_mac"] and original["dst_mac"] != clean["dst_mac"]
    for original, clean in zip(filtered_packets, sanitized_packets)
)
payloads_changed = all(a["payload"] != b["payload"] for a, b in zip(filtered_packets, sanitized_packets))
assert size_fidelity and time_fidelity and order_fidelity and flow_fidelity
assert endpoints_changed and payloads_changed
fidelity = {
    "packet_count": len(sanitized_packets), "sizes_exact": size_fidelity,
    "timestamps_exact": time_fidelity, "order_exact": order_fidelity,
    "flow_distribution_exact": flow_fidelity, "checksums_valid": True,
    "endpoints_changed": endpoints_changed, "payloads_changed": payloads_changed,
}
fidelity


## 4. Décrire le replay sans le rendre exécutable

Un plan de replay utile peut décrire *quoi vérifier* sans fournir d'interface, de cible, de commande ni de permission. Le JSON ci-dessous conserve des offsets relatifs et des empreintes pour la revue, mais impose `execution_allowed=false` et une autorisation non accordée.

Ce **plan inerte** est un artefact de préparation et de discussion. Le transformer en émission réseau demanderait une nouvelle procédure : propriétaire du banc, fenêtre, isolation, effets attendus, arrêt d'urgence, observabilité et approbation formelle. Le notebook ne réalise aucune de ces actions.


In [ ]:
first_timestamp = sanitized_packets[0]["timestamp_us"]
replay_plan = {
    "schema": "agilab.inert-replay-plan.v1",
    "execution_allowed": False,
    "artifact_class": "synthetic-pedagogical",
    "authorization": {
        "required": True,
        "granted": False,
        "scope": None,
        "approver": None,
    },
    "live_execution": {
        "interface": None,
        "target": None,
        "command": None,
        "network_access": False,
    },
    "source": {
        "kind": "sanitized synthetic PCAP",
        "sha256": hashlib.sha256(sanitized_pcap_bytes).hexdigest(),
        "packet_count": len(sanitized_packets),
    },
    "timing": {"basis": "relative-microseconds", "scale": 1.0},
    "review_packets": [
        {
            "ordinal": index,
            "offset_us": packet["timestamp_us"] - first_timestamp,
            "captured_length": packet["frame_length"],
            "frame_sha256": hashlib.sha256(packet["frame"]).hexdigest(),
        }
        for index, packet in enumerate(sanitized_packets)
    ],
}
plan_json = json.dumps(replay_plan, ensure_ascii=False, indent=2, sort_keys=True)
assert replay_plan["execution_allowed"] is False
assert replay_plan["authorization"]["granted"] is False
assert all(value is None for value in (
    replay_plan["live_execution"]["interface"],
    replay_plan["live_execution"]["target"],
    replay_plan["live_execution"]["command"],
))
assert [item["offset_us"] for item in replay_plan["review_packets"]] == [
    packet["timestamp_us"] - first_timestamp for packet in sanitized_packets
]
{"execution_allowed": replay_plan["execution_allowed"], "review_packets": len(replay_plan["review_packets"])}


## 5. Lire les preuves, puis exporter les artefacts inertes

Les deux lignes doivent avoir les mêmes positions et tailles paquet par paquet. Une couleur représente un flow logique. Les traits verticaux relient l'original filtré au plan de replay ; ils ne représentent aucune transmission.

Téléchargez le PCAP assaini et le JSON pour les examiner hors ligne. Ces liens sont des `data:` URI construits en mémoire : ils n'ouvrent aucune interface et n'envoient aucun paquet.


In [ ]:
class RichCaptureReplay:
    def __init__(self, content: str):
        self.content = content

    def _repr_html_(self) -> str:
        return self.content

timeline_start = filtered_packets[0]["timestamp_us"]
timeline_span = filtered_packets[-1]["timestamp_us"] - timeline_start
flow_colors = {5000: "#38bdf8", 6000: "#f59e0b"}
original_marks = []
replay_marks = []
connectors = []
for original, clean in zip(filtered_packets, sanitized_packets):
    x = 95 + 790 * (original["timestamp_us"] - timeline_start) / timeline_span
    width = 4 + original["frame_length"] / 20
    color = flow_colors[original["dst_port"]]
    original_marks.append(f'<rect x="{x:.1f}" y="226" width="{width:.1f}" height="18" rx="4" fill="{color}"/>')
    replay_marks.append(f'<rect x="{x:.1f}" y="326" width="{width:.1f}" height="18" rx="4" fill="{color}"/>')
    connectors.append(f'<line x1="{x + width/2:.1f}" y1="246" x2="{x + width/2:.1f}" y2="324" stroke="#64748b" stroke-dasharray="3 5" opacity=".45"/>')

pcap_data_uri = "data:application/vnd.tcpdump.pcap;base64," + base64.b64encode(sanitized_pcap_bytes).decode("ascii")
plan_data_uri = "data:application/json;base64," + base64.b64encode(plan_json.encode("utf-8")).decode("ascii")
pcap_digest = hashlib.sha256(sanitized_pcap_bytes).hexdigest()
plan_digest = hashlib.sha256(plan_json.encode("utf-8")).hexdigest()
svg = f'''
<svg viewBox="0 0 980 470" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Preuves de fidélité du PCAP assaini et du plan inerte">
  <defs><linearGradient id="bg" x1="0" x2="1"><stop stop-color="#071426"/><stop offset="1" stop-color="#132b46"/></linearGradient></defs>
  <rect width="980" height="470" rx="24" fill="url(#bg)"/>
  <text x="42" y="52" fill="#f8fafc" font-size="25" font-family="system-ui" font-weight="700">Capture → preuve → plan inerte</text>
  <text x="42" y="80" fill="#94a3b8" font-size="14" font-family="system-ui">Aucun paquet transmis • checksums IPv4/UDP vérifiés • seed {SEED}</text>
  <g font-family="system-ui">
    <rect x="42" y="105" width="205" height="72" rx="13" fill="#123451"/><text x="60" y="134" fill="#94a3b8" font-size="12">PAQUETS RETENUS</text><text x="60" y="163" fill="#f8fafc" font-size="27" font-weight="700">{len(filtered_packets)} / {len(parsed_packets)}</text>
    <rect x="266" y="105" width="205" height="72" rx="13" fill="#123451"/><text x="284" y="134" fill="#94a3b8" font-size="12">FIDÉLITÉ</text><text x="284" y="163" fill="#4ade80" font-size="25" font-weight="700">4 / 4 exactes</text>
    <rect x="490" y="105" width="205" height="72" rx="13" fill="#123451"/><text x="508" y="134" fill="#94a3b8" font-size="12">IDENTITÉS + PAYLOAD</text><text x="508" y="163" fill="#4ade80" font-size="25" font-weight="700">remplacés</text>
    <rect x="714" y="105" width="224" height="72" rx="13" fill="#3b1d2a"/><text x="732" y="134" fill="#fda4af" font-size="12">EXECUTION_ALLOWED</text><text x="732" y="163" fill="#fb7185" font-size="25" font-weight="700">false</text>
    <text x="42" y="240" fill="#cbd5e1" font-size="13" font-weight="700">ORIGINAL FILTRÉ</text>
    <line x1="95" y1="250" x2="900" y2="250" stroke="#64748b"/>
    {''.join(connectors)}{''.join(original_marks)}
    <text x="42" y="340" fill="#cbd5e1" font-size="13" font-weight="700">PLAN INERTE</text>
    <line x1="95" y1="350" x2="900" y2="350" stroke="#64748b"/>
    {''.join(replay_marks)}
    <circle cx="65" cy="397" r="6" fill="#38bdf8"/><text x="78" y="402" fill="#cbd5e1" font-size="12">télémétrie : 14</text>
    <circle cx="222" cy="397" r="6" fill="#f59e0b"/><text x="235" y="402" fill="#cbd5e1" font-size="12">commandes : 8</text>
    <text x="42" y="438" fill="#94a3b8" font-size="12">PCAP {pcap_digest[:12]} • plan {plan_digest[:12]} • tailles, temps, ordre, flows exacts</text>
  </g>
</svg>
'''
dashboard_html = f'''
<div style="font-family:system-ui;max-width:980px">{svg}
<div style="display:flex;gap:12px;margin-top:12px;flex-wrap:wrap">
  <a download="capture_assainie_synthetique.pcap" href="{pcap_data_uri}" style="padding:10px 14px;background:#075985;color:white;border-radius:9px;text-decoration:none">⬇ PCAP assaini</a>
  <a download="replay_plan_inerte.json" href="{plan_data_uri}" style="padding:10px 14px;background:#7c2d12;color:white;border-radius:9px;text-decoration:none">⬇ Plan JSON inerte</a>
</div></div>
'''
assert "<svg" in dashboard_html and "execution_allowed" in plan_json
RichCaptureReplay(dashboard_html)


## 6. Restitution, grille de preuve et limites

Expliquez en trois phrases : (1) pourquoi les checksums doivent changer, (2) ce que signifie ici la fidélité, (3) pourquoi le JSON n'est pas une autorisation de replay. Puis identifiez une information encore potentiellement ré-identifiante malgré l'anonymisation.

Barème conseillé sur 10 : format PCAP et encapsulation (2), filtre justifié (1), anonymisation et recalcul des checksums (2), preuves de fidélité (2), distinction capture/replay et autorisation (2), limites correctement formulées (1).

### Limites et sécurité

- capture entièrement synthétique et en mémoire : aucune interface, aucun sniffing, aucun socket, aucun trafic terrain ;
- couverture limitée à Ethernet II, IPv4 sans options et UDP ; pas de VLAN, IPv6, TCP, fragmentation, chiffrement ni paquet tronqué accepté ;
- les timestamps PCAP sont préservés à la microseconde, mais cela ne démontre pas qu'un système réel pourrait respecter ce timing ;
- la fidélité prouvée porte sur les tailles, l'horodatage, l'ordre et la distribution des flows, pas sur la sémantique applicative ;
- conserver timing, tailles et ports peut permettre une ré-identification : pseudonymiser n'est pas anonymiser de façon absolue ;
- le plan reste strictement inerte : cible, interface, commande, approbateur et périmètre sont absents ;
- un replay réel pourrait déclencher une commande, surcharger un équipement ou violer une politique. Il exige un banc isolé, une autorisation explicite et distincte, une revue sécurité et un mécanisme d'arrêt.

**Frontière de preuve :** ce TP apprend à raisonner sur une chaîne capture → assainissement → vérification → préparation. Il ne constitue ni une procédure opérationnelle de capture, ni un outil de replay, ni une preuve de sécurité en production.
